In [ ]:
%cd ..

# TICA Landscape

In [ ]:
from dataclasses import dataclass
import pickle
import numpy as np
from deeptime.decomposition import TICA

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    print("joblib not found, parallel processing disabled")

@dataclass(slots=True, frozen=True)
class NBConfig:
    # TICA settings
    n_components: int = 2
    lagtime_start_fraction: float = 0.01  # fraction of n_frames
    lagtime_end_fraction: float = 0.1
    lagtime_step_fraction: float = 0.01

In [ ]:
def _compute_tica(data: np.ndarray, lagtime: int, config: NBConfig):
    try:
        tica = TICA(dim=config.n_components, lagtime=lagtime).fit_fetch(data)
        singular_values = tica.singular_values[:config.n_components] # type: ignore
        timescales = tica.timescales(lagtime=lagtime)[:config.n_components] # type: ignore
        vamp2 = (tica.singular_values ** 2).sum() # type: ignore

    except Exception:
        singular_values = np.full(config.n_components, np.nan)
        timescales = np.full(config.n_components, np.nan)
        vamp2 = np.nan 

    return {
        "singular_values": singular_values,
        "timescales": timescales,
        "vamp2": vamp2
    }

def _compute_tica_landscape(data: np.ndarray, config: NBConfig):
    flat_data = np.reshape(data, (data.shape[0], -1))

    # Compute lag times
    n_frames = data.shape[0]
    lagtimes = np.arange(
        int(n_frames * config.lagtime_start_fraction),
        int(n_frames * config.lagtime_end_fraction),
        int(n_frames * config.lagtime_step_fraction)
    )

    if HAS_JOBLIB:
        tica_landscape = Parallel(n_jobs=-1, verbose=10)( # type: ignore
            delayed(_compute_tica)(flat_data, lagtime, config) # type: ignore
            for lagtime in lagtimes
        )

    else:
        tica_landscape = [
            _compute_tica(flat_data, lagtime, config) # type: ignore
            for lagtime in lagtimes
        ]

    return lagtimes, tica_landscape

def compute_tica_landscapes(data: dict[str, np.ndarray], config: NBConfig):
    """Compute TICA landscapes"""
    results = {}
    for representation, components in data.items():
        result = _compute_tica_landscape(components, config)
        results[representation] = result
    return results

In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

config = NBConfig(n_components=2)

In [ ]:
with open("examples/pcs_preprocessing.pkl", "rb") as h:
    components = pickle.load(h)["data"]

stats = compute_tica_landscapes(components, config)

with open("examples/tica_landscape.pkl", "wb") as h:
    pickle.dump({
        "stats": stats
    }, h, protocol=pickle.HIGHEST_PROTOCOL)